# Pipeline EEG → graphe de connectivité cérébrale

Ce notebook exécute **toute** la chaîne, du fichier EEG brut au graphe et à ses métriques. Le code vit dans
le package `eeg_graph` (`src/eeg_graph/`) ; le notebook ne fait qu'enchaîner les étapes et afficher les
contrôles.

Jeu de test : PhysioNet **EEGMMIDB**, R01 (repos yeux ouverts) et R02 (repos yeux fermés). Il sert à
**valider la chaîne** (effet Berger attendu) et ne fixe pas le use case.

| Section | Contenu |
|---|---|
| 0 | Configuration et paramètres |
| 1 | Chargement et contrôles des runs |
| 2 | Prétraitement : filtrage, référence moyenne, ICA, rejet d'epochs |
| 3 | Nœuds : 68 régions Desikan-Killiany via le noyau inverse W (contrôle V5) |
| 4 | Connectivité par bande (wPLI, ciPLV, AEC-orth) et cache |
| 5 | Graphe et métriques exigées (taille, connexité, diamètre/rayon, centralités, communautés, Laplacien) |
| 6 | Batch multi-sujets |
| 7 | Validation (V1, V2, V4, V6) |
| 8 | Mode capteurs (nœuds = électrodes) |

Exécution : noyau Python du projet (`uv sync`, puis sélectionner `.venv` comme noyau). Les données
(~1 Mo par run, fsaverage ~760 Mo) sont téléchargées automatiquement dans `data/` au premier passage.

## 0. Configuration

In [ ]:
%matplotlib inline
import json
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from eeg_graph import validation
from eeg_graph.analysis import REQUIRED_COLUMNS, community_agreement, graph_from_result, metrics_table, node_table
from eeg_graph.config import REST_4S, REST_12S, SEED, EEGMMIDBSpec, Paths, PipelineConfig, setup_logging
from eeg_graph.metrics import compute_metrics
from eeg_graph.pipeline import build_cache, compute_subject_results, make_node_model, prepare_eegmmidb_subject
from eeg_graph.source import fsaverage_mesh
from eeg_graph.viz import (
    plot_brain_graph_3d,
    plot_brain_graph_views,
    plot_circular_graph,
    plot_connectivity_matrix,
    plot_region_values,
)

setup_logging()
paths, cfg = Paths(), PipelineConfig()
conditions = EEGMMIDBSpec().conditions
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
paths.figures.mkdir(parents=True, exist_ok=True)
TABLES = paths.root / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    fig.savefig(paths.figures / name, dpi=200, bbox_inches="tight")


print("Configuration", cfg.hash(), "|", cfg.inverse)
print(cfg.preproc)

**Paramètres de l'exécution.** `NODE_SPACE = "source"` donne les 68 régions corticales ; `"sensor"`
utilise les électrodes comme nœuds. Les runs 1 et 2 sont nécessaires pour la section 7.

In [ ]:
SUBJECT = 1  # sujet de référence (sections 1 à 5 et 8)
RUNS = [1, 2]  # R01 yeux ouverts, R02 yeux fermés
NODE_SPACE = "source"
METRIC, BAND = "wpli", "alpha"  # graphe présenté en détail
DENSITIES = [0.10, 0.15, 0.20]  # seuillage proportionnel
DENSITY = 0.15
BATCH_SUBJECTS = list(range(1, 11))  # section 6 (le batch complet attend le choix du use case)
N_JOBS = 4  # 4 workers max sur 8 Go de RAM
OVERWRITE = False

## 1. Chargement et contrôles

Chaque run est contrôlé : fréquence d'échantillonnage (160 Hz), 64 canaux EEG, durée, et pour les runs de
tâche la durée des essais. Un run qui échoue est exclu et l'exclusion est consignée dans `index.csv`. Les
données de S088, S092 et S100 sont à 128 Hz sur leurs runs de tâche, par exemple.

In [ ]:
prepared = prepare_eegmmidb_subject(SUBJECT, RUNS, cfg, paths)
pd.DataFrame([asdict(c) for c in prepared.checks.values()])

Contrôle de bon sens du signal : en yeux fermés, le pic alpha (8-13 Hz) des électrodes occipitales doit
être nettement plus haut qu'en yeux ouverts.

In [ ]:
occipital = ["O1", "Oz", "O2", "PO7", "PO8"]
fig, ax = plt.subplots(figsize=(7, 3.5))
for run, raw in prepared.cleaned.items():
    psd = raw.compute_psd(fmin=1, fmax=45, picks=occipital, verbose="error")
    ax.semilogy(psd.freqs, psd.get_data().mean(axis=0) * 1e12, label=f"R{run:02d} {conditions[run]}")
ax.axvspan(8, 13, color="0.9", zorder=0)
ax.set(xlabel="fréquence (Hz)", ylabel="DSP (µV²/Hz)", title=f"S{SUBJECT:03d}, électrodes occipitales, après ICA")
ax.legend()
save(fig, f"sub-{SUBJECT:03d}_psd_occipital.png")

## 2. Prétraitement

- Filtre FIR zero-phase 0.5-45 Hz. Pas de notch : le passe-bas coupe déjà au-delà de 56 Hz.
- Référence moyenne en projection, requise pour la modélisation inverse.
- **ICA picard, une par sujet** (20 composantes), ajustée sur les runs concaténés et filtrés à 1 Hz.
  Faute de canal EOG, Fp1/Fp2 servent de proxy pour repérer les composantes oculaires (3 au plus).

In [ ]:
ica = prepared.ica
excluded = list(ica.excluded)
print(f"Composantes retirées : {excluded} ; |r| max avec Fp1/Fp2 : {ica.eog_scores[excluded].round(2)}")
others = np.sort(np.delete(ica.eog_scores, excluded))[::-1]
print("Scores des composantes suivantes :", others[:3].round(2))
ica.ica.plot_components(picks=range(10), show=False);

**Rejet d'epochs.** Le brief proposait un seuil fixe de 150 µV crête-à-crête. Sur ces données, il rejette
toutes les epochs, et l'alpha yeux fermés atteint 400-500 µV en occipital : un seuil fixe plus haut
rejetterait davantage la condition yeux fermés. On rejette donc les epochs **aberrantes pour
l'enregistrement** : z-score robuste supérieur à 3 de l'amplitude crête-à-crête maximale, inspiré de
FASTER (Nolan et al. 2010), avec un plafond absolu de 1 mV. Les barres rouges marquent les epochs rejetées.

In [ ]:
ptp = validation.epoch_ptp_table(prepared, REST_4S, cfg)
fig, axes = plt.subplots(1, len(prepared.cleaned), figsize=(6 * len(prepared.cleaned), 3.2), sharey=True, squeeze=False)
for ax, (run, df) in zip(axes[0], ptp.groupby("run"), strict=True):
    ax.bar(df.epoch - 0.2, df.ptp_before_uV, 0.4, color="0.7", label="avant ICA")
    ax.bar(df.epoch + 0.2, df.ptp_after_uV, 0.4, color=["C0" if k else "red" for k in df.kept], label="après ICA")
    ax.axhline(150, ls="--", c="k", lw=0.8, label="150 µV (seuil fixe du brief)")
    ax.set(title=f"R{run:02d} {conditions[run]}", xlabel="epoch de 4 s", ylabel="crête-à-crête max (µV)")
axes[0, 0].legend(fontsize=7)
save(fig, f"sub-{SUBJECT:03d}_ptp.png")
ptp.groupby("run").kept.agg(retenues="sum", total="size")

## 3. Nœuds : du capteur à la région corticale

En mode `source`, le modèle direct est calculé sur le template **fsaverage** (BEM 3 couches, espace source
ico-5) et l'inverse est **eLORETA** (λ² = 1, orientation normale, covariance de bruit ad hoc). Les sources
sont ensuite moyennées dans les 68 régions Desikan-Killiany (`mean_flip`).

Le template, la covariance et les électrodes étant communs à tous les enregistrements, l'opérateur est
identique pour tous. Comme l'inverse et `mean_flip` sont linéaires, toute cette étape se réduit à une
matrice fixe $W \in \mathbb{R}^{68 \times 64}$ : séries des régions $= W X$. Le premier passage calcule $W$
(~20 s), qui est ensuite lu depuis le cache.

In [ ]:
nodes = make_node_model(prepared.info, NODE_SPACE, cfg, paths)
atlas = nodes.atlas
atlas_df = pd.DataFrame({"node": atlas.names, "hemisphere": atlas.hemispheres, "lobe": atlas.lobes}).join(
    pd.DataFrame(atlas.coords_mm.round(1), columns=["x_mm", "y_mm", "z_mm"])
)
display(atlas_df.groupby(["hemisphere", "lobe"]).size().unstack(fill_value=0))
atlas_df.head()

**V5 : équivalence du noyau.** $W X$ doit reproduire exactement `apply_inverse_epochs` +
`extract_label_time_course`, la chaîne MNE complète.

In [ ]:
if NODE_SPACE == "source":
    err = validation.kernel_equivalence(prepared.cleaned[RUNS[0]], cfg.inverse, paths)
    print(f"V5 : écart relatif max entre W @ X et la chaîne MNE = {err:.1e}")
    assert err < 1e-6

## 4. Connectivité par bande et cache

Pour chaque bande (δ 1-4, θ 4-8, α 8-13, β 13-30, γ 30-45 Hz), les séries des nœuds sont filtrées **en
continu** (FIR), puis transformées par Hilbert et découpées aux epochs retenues. Sur chaque epoch, on
calcule trois métriques insensibles au couplage à déphasage nul (conduction de volume, fuite de sources) :

- **wPLI** (Vinck et al. 2011) : $\frac{|\sum_t \Im S_{ij}(t)|}{\sum_t |\Im S_{ij}(t)|}$, avec $S_{ij} = z_i \bar z_j$ ;
- **ciPLV** (Bruña et al. 2018) : $\frac{|\Im M|}{\sqrt{1 - (\Re M)^2}}$, avec $M = \langle e^{i(\varphi_i - \varphi_j)} \rangle_t$ ;
- **AEC orthogonalisée** (Hipp et al. 2012), via mne-connectivity.

La matrice d'un enregistrement est la moyenne de ses epochs. Chaque enregistrement est écrit dans un `.npz`
(`data/cache/eegmmidb/<mode>/`), avec un `index.csv` global. On calcule aussi un découpage 5 × 12 s, celui
du protocole de Fraschini et al. (2021).

In [ ]:
results = compute_subject_results(prepared, nodes, NODE_SPACE, (REST_12S,), cfg, paths)
main = [r for r in results if r.epoch_spec != REST_12S.name]
pd.DataFrame(
    [
        {k: r.meta()[k] for k in ("run", "condition", "epoch_spec", "n_epochs", "n_epochs_total", "seconds")}
        for r in results
    ]
)

In [ ]:
fig, axes = plt.subplots(len(main), len(cfg.metrics), figsize=(5.3 * len(cfg.metrics), 4.7 * len(main)), squeeze=False)
for j, metric in enumerate(cfg.metrics):
    vmax = max(float(r.matrix(metric, BAND).max()) for r in main)
    for i, r in enumerate(main):
        plot_connectivity_matrix(
            axes[i, j], r.matrix(metric, BAND), atlas, f"R{r.run:02d} {r.condition} — {metric} {BAND}", vmax
        )
save(fig, f"sub-{SUBJECT:03d}_{NODE_SPACE}_matrices_{BAND}.png")

## 5. Du graphe aux métriques exigées

**Modèle de graphe :**
- graphe **non orienté** (les métriques sont symétriques), **simple** (une arête par paire, pas de boucle)
  et **pondéré** (poids = connectivité dans [0, 1]) ; nœuds = régions, avec hémisphère, lobe et centroïde MNI ;
- **seuillage proportionnel** : on garde les 10, 15 ou 20 % de paires les plus fortes (228, 342 ou 456 arêtes
  sur 2 278), ou le graphe pondéré complet ;
- chemins pondérés avec une longueur d'arête de $1/w$.

**Conventions :**
- diamètre et rayon en nombre de sauts, sur la plus grande composante si le graphe n'est pas connexe
  (`distances_on`). Sur le graphe pondéré complet, ils valent trivialement 1 : seuls les graphes seuillés
  sont informatifs pour les distances ;
- efficacité globale et clustering pondérés, comme dans Fraschini et al. (2021) ;
- Louvain répété sur 100 seeds, en gardant la partition de modularité maximale.

In [ ]:
table = metrics_table(main, METRIC, BAND, [None, *DENSITIES])
table.to_csv(TABLES / f"sub-{SUBJECT:03d}_{NODE_SPACE}_{METRIC}_{BAND}_metrics.csv", index=False)
table[["run", "condition", "threshold", *REQUIRED_COLUMNS]]

**Centralités** du graphe yeux fermés au seuil `DENSITY` : quels nœuds sont des hubs ? Le degré et la
strength mesurent la connectivité locale, la betweenness le rôle de relais sur les plus courts chemins, la
closeness la proximité au reste du réseau, et l'eigenvector la connexion à d'autres nœuds centraux.

In [ ]:
r_ec = main[-1]
g_ec = graph_from_result(r_ec, METRIC, BAND, DENSITY)
m_ec = compute_metrics(g_ec, seed=SEED)
nodes_ec = node_table(r_ec, m_ec)
nodes_ec.to_csv(TABLES / f"sub-{SUBJECT:03d}_run-{r_ec.run:02d}_{NODE_SPACE}_{METRIC}_{BAND}_nodes.csv", index=False)
display(nodes_ec.sort_values("betweenness", ascending=False).head(10))
nodes_ec.groupby("lobe")[["strength", "betweenness", "eigenvector"]].mean().sort_values("strength", ascending=False)

**Communautés :** est-ce que la partition de Louvain suit l'anatomie ? On la compare aux lobes et aux
hémisphères avec la NMI et l'ARI (0 = indépendante de l'anatomie, 1 = identique).

In [ ]:
rows = []
for r in main:
    m = compute_metrics(graph_from_result(r, METRIC, BAND, DENSITY), seed=SEED)
    rows.append(
        {
            "run": r.run,
            "condition": r.condition,
            "n_communities": m.n_communities,
            "modularity": m.modularity,
            **community_agreement(r, m),
        }
    )
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, len(main), figsize=(5.6 * len(main), 10.5), squeeze=False)
vmax = max(float(r.matrix(METRIC, BAND).max()) for r in main)
for col, r in enumerate(main):
    plot_connectivity_matrix(
        axes[0, col],
        r.matrix(METRIC, BAND),
        atlas,
        f"S{SUBJECT:03d} R{r.run:02d} {r.condition} — {METRIC} {BAND}",
        vmax,
    )
    g = graph_from_result(r, METRIC, BAND, DENSITY)
    m = compute_metrics(g, seed=SEED)
    plot_circular_graph(
        axes[1, col],
        g,
        atlas,
        m.community,
        m.betweenness,
        f"{round(DENSITY * 100)} % des arêtes — {m.n_communities} communautés (Q = {m.modularity:.2f})",
    )
save(fig, f"sub-{SUBJECT:03d}_{NODE_SPACE}_{METRIC}_{BAND}_d{round(DENSITY * 100)}.png")

**Spectre du Laplacien normalisé.** La multiplicité de la valeur propre 0 est égale au nombre de composantes
connexes. La connectivité algébrique $\lambda_2$ du Laplacien combinatoire mesure à quel point le graphe
est difficile à couper en deux.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for r in main:
    m = compute_metrics(graph_from_result(r, METRIC, BAND, DENSITY), seed=SEED, n_louvain_runs=10)
    label = f"R{r.run:02d} {r.condition} : {m.n_zero_laplacian_eigs} valeur(s) propre(s) nulle(s)"
    ax.plot(m.laplacian_eigenvalues, ".", ms=4, label=f"{label}, λ₂ = {m.algebraic_connectivity:.3f}")
ax.set(xlabel="rang", ylabel="valeur propre", title=f"Laplacien normalisé — {METRIC} {BAND}, {round(DENSITY * 100)} %")
ax.legend(fontsize=8)
save(fig, f"sub-{SUBJECT:03d}_{NODE_SPACE}_{METRIC}_{BAND}_laplacian.png")

**Graphe non connexe.** Au seuil proportionnel, le graphe peut se fragmenter. Seconde convention : l'union
avec l'arbre couvrant de poids maximal garantit la connexité, au prix de quelques arêtes en plus. Les
distances sont alors définies sur tout le graphe.

In [ ]:
connected = metrics_table(main, METRIC, BAND, DENSITIES, ensure_connected=True, n_louvain_runs=20)
connected[
    [
        "run",
        "condition",
        "threshold",
        "n_edges",
        "is_connected",
        "n_components",
        "diameter",
        "radius",
        "global_efficiency_weighted",
    ]
]

**Le graphe dans le cerveau.** Mêmes graphe et métriques que ci-dessus (yeux fermés, seuil `DENSITY`) : nœuds aux centroïdes MNI des régions, dans la surface piale de fsaverage ; taille = strength, couleur = communauté de Louvain, épaisseur des arêtes = poids. La figure interactive (rotation, zoom, survol des nœuds) est aussi enregistrée en HTML dans `reports/figures/` ; les trois projections servent au rapport. En mode `sensor`, les électrodes sont tracées sans cerveau (repère tête, pas MNI).

In [ ]:
mesh = fsaverage_mesh(paths) if NODE_SPACE == "source" else None
title = f"S{SUBJECT:03d} R{r_ec.run:02d} {r_ec.condition} — {METRIC} {BAND}, {round(DENSITY * 100)} % des arêtes"
stem = f"sub-{SUBJECT:03d}_run-{r_ec.run:02d}_{NODE_SPACE}_{METRIC}_{BAND}_d{round(DENSITY * 100)}_brain"
fig = plot_brain_graph_views(g_ec, atlas, mesh, m_ec.community, m_ec.strength, title)
save(fig, f"{stem}.png")
plt.show()
fig3d = plot_brain_graph_3d(g_ec, atlas, mesh, m_ec.community, m_ec.strength, title)
fig3d.write_html(paths.figures / f"{stem}.html", include_plotlyjs="cdn")  # plotly.js chargé depuis le CDN
fig3d.show()

## 6. Batch multi-sujets

Même chaîne sur `BATCH_SUBJECTS`. Le calcul est idempotent : un sujet déjà en cache avec le même hash de
configuration est sauté. Le batch complet (109 sujets, runs du use case) attend le choix du use case.

In [ ]:
summary = build_cache(BATCH_SUBJECTS, RUNS, NODE_SPACE, n_jobs=N_JOBS, overwrite=OVERWRITE)
display(summary)
index = pd.read_csv(paths.cache / "eegmmidb" / "index.csv")
index.groupby(["node_space", "epoch_spec", "status"]).size().rename("n").reset_index()

## 7. Validation de la chaîne

- **V1 (bloquant)** : effet Berger en source. Le ratio de puissance alpha EC/EO doit être maximal dans les
  régions pariéto-occipitales, pour le sujet de référence et en moyenne sur le batch. Contrôle secondaire :
  le bloc wPLI alpha postérieur doit être plus fort en yeux fermés.
- **V2** : toutes les matrices du cache sont symétriques, à diagonale nulle, finies et dans [0, 1].
- **V4** : temps de calcul par enregistrement.
- **V6** : sensibilité au choix de l'inverse (eLORETA, dSPM, MNE × λ²).

In [ ]:
assert {1, 2} <= set(RUNS), "la validation compare R01 et R02"
report = {
    "V2": {"files_checked": validation.validate_cache(paths), "passed": True},
    "V4": validation.timing_summary(paths, NODE_SPACE),
}
pairs = validation.load_rest_pairs(paths, BATCH_SUBJECTS, NODE_SPACE)
eo, ec = pairs[SUBJECT]
report["V1_subject"] = validation.berger_check(validation.alpha_log_power_ratio(eo, ec), atlas)
group_ratio = np.mean([validation.alpha_log_power_ratio(a, b) for a, b in pairs.values()], axis=0)
report["V1_group"] = {"n_subjects": len(pairs), **validation.berger_check(group_ratio, atlas)}
report["V1_wpli_posterior_block"] = validation.wpli_posterior_block_test(pairs)
print(json.dumps(report, indent=1, ensure_ascii=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
plot_region_values(axes[0], group_ratio, atlas, f"log10 P_EC / P_EO (alpha), moyenne de {len(pairs)} sujets")
mean_eo = np.mean([a.matrix("wpli", "alpha") for a, _ in pairs.values()], axis=0)
mean_ec = np.mean([b.matrix("wpli", "alpha") for _, b in pairs.values()], axis=0)
vmax = float(max(mean_eo.max(), mean_ec.max()))
plot_connectivity_matrix(axes[1], mean_eo, atlas, f"wPLI alpha EO, moyenne de {len(pairs)} sujets", vmax)
plot_connectivity_matrix(axes[2], mean_ec, atlas, f"wPLI alpha EC, moyenne de {len(pairs)} sujets", vmax)
save(fig, f"validation_V1_{NODE_SPACE}.png")

**V6 : sensibilité à l'inverse** (sujet de référence, R02). On compare les matrices wPLI alpha obtenues
avec chaque variante par corrélation de Spearman des triangles supérieurs, et on relève le bloc postérieur
moyen. Sur S001, à λ² égal, les méthodes concordent ; changer λ² modifie le classement fin des arêtes, mais
l'effet agrégé (bloc postérieur) reste stable. Il faut donc reporter la sensibilité à λ² pour le use case.

In [ ]:
if NODE_SPACE == "source":
    corr, blocks = validation.inverse_robustness(prepared.cleaned[2], cfg, paths)
    fig, ax = plt.subplots(figsize=(7.5, 6.3))
    im = ax.imshow(corr.to_numpy(), vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(corr)), corr.columns, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(corr)), corr.index, fontsize=8)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(
                j,
                i,
                f"{corr.iat[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=6.5,
                color="w" if corr.iat[i, j] < 0.6 else "k",
            )
    plt.colorbar(im, ax=ax, label="ρ de Spearman")
    ax.set_title(f"V6 : wPLI alpha, S{SUBJECT:03d} R02")
    save(fig, f"validation_V6_sub-{SUBJECT:03d}.png")
    corr.round(3).to_csv(TABLES / f"v6_inverse_spearman_sub-{SUBJECT:03d}.csv")
    report["V6"] = {"spearman": corr.round(3).to_dict(), "posterior_block": blocks.to_dict(orient="records")}
    display(blocks.round(3))

In [ ]:
out = paths.root / "reports" / f"validation_{NODE_SPACE}.json"
out.write_text(json.dumps(report, indent=2, ensure_ascii=False))
verdicts = {
    "V1 sujet": report["V1_subject"]["passed"],
    "V1 groupe": report["V1_group"]["passed"],
    "V1 bloc wPLI (p < 0.05)": (report["V1_wpli_posterior_block"]["wilcoxon_p"] or 1) < 0.05,
    "V2 cache valide": report["V2"]["passed"],
}
pd.Series(verdicts, name="validé")

## 8. Mode capteurs

Même sujet, avec les électrodes comme nœuds (64 nœuds, sans modélisation inverse). C'est le repli pour les
datasets peu denses ou en montage bipolaire, où l'espace source n'est pas défendable.

In [ ]:
nodes_sensor = make_node_model(prepared.info, "sensor", cfg, paths)
results_sensor = compute_subject_results(prepared, nodes_sensor, "sensor", (), cfg, paths)
fig, axes = plt.subplots(1, len(results_sensor), figsize=(5.5 * len(results_sensor), 4.7), squeeze=False)
vmax = max(float(r.matrix(METRIC, BAND).max()) for r in results_sensor)
for ax, r in zip(axes[0], results_sensor, strict=True):
    plot_connectivity_matrix(
        ax, r.matrix(METRIC, BAND), nodes_sensor.atlas, f"capteurs R{r.run:02d} {r.condition} — {METRIC} {BAND}", vmax
    )
save(fig, f"sub-{SUBJECT:03d}_sensor_matrices_{BAND}.png")
metrics_table(results_sensor, METRIC, BAND, [DENSITY])[["run", "condition", "threshold", *REQUIRED_COLUMNS]]

## Sorties

- `data/cache/eegmmidb/` : matrices de connectivité (`.npz`) et `index.csv` ;
- `reports/figures/` : figures de ce notebook ;
- `reports/tables/` : tableaux de métriques (globales, nodales, V6) ;
- `reports/validation_<mode>.json` : résultats de validation.

Les tests unitaires des fonctions pures restent dans `tests/` (`uv run pytest`).